In [1]:
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl
import alphalens as al
import matplotlib.pyplot as plt
from datetime import datetime

# 配置文件
try:
    import config_local as config
except ImportError:
    import config

# 导入数据接口sdk
import zenidatasdk as zd
client = zd.Client(
    base_url=config.ZENI_URL,
    username=config.ZENI_USERNAME,
    password=config.ZENI_PASSWORD,
)

# 忽视警告信息
warnings.filterwarnings(action = 'ignore')

In [2]:
# 历史回测区间
init_date = '2025-11-01'
start_date = '2025-12-01'
end_date = str(datetime.today().date())
index_symbol = rf"000852.XSHG"

# rf"000300.XSHG",  # 沪深300
# rf"000905.XSHG",  # 中证500
# rf"000852.XSHG",  # 中证1000
# rf"000016.XSHG",  # 上证50
# rf"399006.XSHE",  # 创业板

In [3]:
# 获取指数成分股数据据
index_weights_df = client.get_index_constituents_df(
    index_symbol=index_symbol,
    start_date=start_date,
    end_date=end_date
)
index_weights_df = index_weights_df.rename(columns={"date": "datetime"})
symbols = index_weights_df["symbol"].unique().tolist()
index_weights_df

,datetime,index_symbol,update_date,symbol,weight,display_name
0,2025-12-01,000852.XSHG,2025-11-28,000012.XSHE,0.069,南玻A
1,2025-12-01,000852.XSHG,2025-11-28,000019.XSHE,0.030,深粮控股
2,2025-12-01,000852.XSHG,2025-11-28,000025.XSHE,0.050,特力A
3,2025-12-01,000852.XSHG,2025-11-28,000028.XSHE,0.061,国药一致
4,2025-12-01,000852.XSHG,2025-11-28,000029.XSHE,0.083,深深房A
...,...,...,...,...,...,...
95995,2026-04-24,000852.XSHG,2026-03-31,688776.XSHG,0.055,国光电气
95996,2026-04-24,000852.XSHG,2026-03-31,688779.XSHG,0.143,五矿新能
95997,2026-04-24,000852.XSHG,2026-03-31,688789.XSHG,0.084,宏华数科
95998,2026-04-24,000852.XSHG,2026-03-31,688798.XSHG,0.073,艾为电子


In [4]:
# ================ 定义本地分钟数据路径 ================
tmp_Dataset_path = rf"D:\tmp_Dataset"
minute_bar_Dataset_path = os.path.join(tmp_Dataset_path, "minute_bar")  # 分钟数据本地路径
os.makedirs(minute_bar_Dataset_path, exist_ok=True)

index_weights_path = os.path.join(minute_bar_Dataset_path, index_symbol)  # 指数成分股
os.makedirs(index_weights_path, exist_ok=True)


# ================ 按月获取分钟数据 ================
def generate_monthly_ranges_dict(start_date, end_date):
    """生成按月切割的日期区间字典，格式为 {YYMM: (月初, 月末)}"""
    start = datetime.strptime(start_date, '%Y-%m-%d').date()
    end = datetime.strptime(end_date, '%Y-%m-%d').date()

    date_dict = {}
    current = start

    import calendar
    while True:
        month_start = current.replace(day=1)
        _, last_day = calendar.monthrange(current.year, current.month)
        month_end = current.replace(day=last_day)

        if month_end > end:  # 不足一个月舍弃
            break

        # 键格式：YYMM（如 2603 表示 2026年03月）
        key = f"{str(current.year)[-2:]}{current.month:02d}"
        date_dict[key] = (month_start.strftime('%Y-%m-%d'), month_end.strftime('%Y-%m-%d'))

        # 移到下个月
        if current.month == 12:
            current = current.replace(year=current.year + 1, month=1, day=1)
        else:
            current = current.replace(month=current.month + 1, day=1)

    return date_dict

date_tuple_list = generate_monthly_ranges_dict(init_date, end_date)
date_tuple_list

{'2511': ('2025-11-01', '2025-11-30'),
 '2512': ('2025-12-01', '2025-12-31'),
 '2601': ('2026-01-01', '2026-01-31'),
 '2602': ('2026-02-01', '2026-02-28'),
 '2603': ('2026-03-01', '2026-03-31')}

In [5]:
def tmp_filter(dataframe):
    # 1. 临时转换为 datetime 用于过滤
    # 2. 过滤后删除临时列，保留原始字符串
    return None
tmp_filter = None


from tqdm import tqdm
big_minute_bar_dict = {}
for month, start_end_tuple in tqdm(
    date_tuple_list.items(), desc = "获取每月分钟数据"
):

    minute_bar_monthly_path = os.path.join(index_weights_path, rf"{month}.parquet")
    if os.path.exists(minute_bar_monthly_path):
        bars_1m_df_tmp = pd.read_parquet(minute_bar_monthly_path)
    else:
        bars_1m_df_tmp = client.get_kline_df(
            symbol = symbols,
            start_date = start_end_tuple[0],
            end_date = start_end_tuple[1],
            frequency = "1m",
            adjust_type = "post",
            market = "cn_stock",
        )
        bars_1m_df_tmp.to_parquet(minute_bar_monthly_path, engine = 'pyarrow',compression = 'snappy')

    # 删除部分数据，减少内存负担 tmp_filter
    if tmp_filter:
        big_minute_bar_dict[month] = tmp_filter(bars_1m_df_tmp)
    else:
        big_minute_bar_dict[month] = bars_1m_df_tmp
    import gc; gc.collect()


# 分钟数据长表格 bars_1m_df
bars_1m_df = pd.concat(list(big_minute_bar_dict.values()))
bars_1m_df.head()

获取每月分钟数据: 100%|██████████| 5/5 [00:02<00:00,  2.44it/s]


,open,high,low,close,volume,symbol,amount,datetime
0,652.72,652.72,641.36,641.36,46146.83,000006.XSHE,29887897.0,2025-11-03 09:31:00
1,641.36,641.88,635.17,635.17,35893.05,000006.XSHE,22902399.0,2025-11-03 09:32:00
2,632.58,635.17,619.67,622.77,54638.38,000006.XSHE,34146136.0,2025-11-03 09:33:00
3,621.74,623.81,615.03,622.77,61855.72,000006.XSHE,38316132.0,2025-11-03 09:34:00
4,622.77,630.00,622.77,623.29,45381.91,000006.XSHE,28385346.0,2025-11-03 09:35:00


In [6]:
import gc; gc.collect()

18

In [7]:
# 获取日频估值数据
fundamental_1d_df = client.get_valuation_df(
    symbols=symbols,
    start_date=start_date,
    end_date=end_date,
    fields="datetime,symbol,market_cap,circulating_market_cap,turnover_ratio,pe_ratio,pb_ratio,dividend_ratio"
)

# 构建市值数据
mkt_cap_name = "market_cap"
market_cap_df = fundamental_1d_df.set_index(["datetime", "symbol"])[[mkt_cap_name]]

# 负数和无穷值 & 对数处理
market_cap_df[mkt_cap_name] = np.where((market_cap_df[mkt_cap_name] <= 0) | (~np.isfinite(market_cap_df[mkt_cap_name])),
                                       0, market_cap_df[mkt_cap_name])
market_cap_df[f"{mkt_cap_name}_log"] = np.log1p(market_cap_df[mkt_cap_name])
market_cap = market_cap_df[f"{mkt_cap_name}_log"]
market_cap.head()

datetime    symbol     
2025-12-01  000012.XSHE    4.985274
            000019.XSHE    4.425104
            000025.XSHE    4.458403
            000028.XSHE    4.946219
            000029.XSHE    5.509942
Name: market_cap_log, dtype: float64

In [8]:
# 获取行业数据
industry_constituents_df = client.get_industry_constituents_composite_df(
    symbols = symbols,
    category = "sw_l1",
    start_date = start_date,
    end_date = end_date
)

# 构建双重索引的行业数据
industries = industry_constituents_df.set_index(["datetime", "symbol"])["industry_name"]
industries

datetime    symbol     
2025-12-01  000019.XSHE    农林牧渔I
            000048.XSHE    农林牧渔I
            000735.XSHE    农林牧渔I
            000930.XSHE    农林牧渔I
            002041.XSHE    农林牧渔I
                           ...  
2026-04-24  600223.XSHG    美容护理I
            600315.XSHG    美容护理I
            603193.XSHG    美容护理I
            603983.XSHG    美容护理I
            605009.XSHG    美容护理I
Name: industry_name, Length: 106877, dtype: object

In [9]:
# 转化并提取日级数据
prices_df = bars_1m_df[bars_1m_df['datetime'] >= start_date].copy()
prices_df = bars_1m_df.rename(columns = {'datetime': 'datetime_min'})
prices_df['datetime'] = prices_df['datetime_min'].dt.normalize()

# 把日级数据透视为符合 alphalens 格式的开盘价表格
# 策略回测以每天收盘完，因子完成计算后，次日以开盘价买入为准
prices_df = prices_df.groupby(['datetime', "symbol"]).agg(open = ('open', 'first')).reset_index()
prices = prices_df.pivot(index = 'datetime', columns = 'symbol', values = 'open')
prices.head().T.head().T

symbol,000006.XSHE,000010.XSHE,000011.XSHE,000012.XSHE,000016.XSHE
datetime,,,,,
2025-11-03,652.72,50.78,50.22,173.89,132.01
2025-11-04,637.75,53.57,48.93,174.25,133.47
2025-11-05,599.53,51.58,47.46,174.25,132.49
2025-11-06,622.77,52.51,48.15,176.81,134.20
2025-11-07,621.22,51.31,47.04,177.18,133.22


In [17]:
def f_0322(bars: pd.DataFrame, f_name: str = 'Average_relative_price_position', roll_days: int = 20) -> pd.DataFrame:
    """
    factor_intro: 衡量股票在价格相对高位停留的时间长短
    category: 高频因子
    category_intro: 收益率分布类
    subcategory:
    subcategory_intro:
    min_period: 20d
    source: 朱剑涛, 2020, 基于时间维度度量的日内买卖压力, 东方证券
    author: 因子团队

    说明：使用每日K线的典型价格（(开盘+收盘+最高+最低)/4）计算日内时间加权平均价格（TWAP），
    并在过去 cal_range 个交易日内按交易时长加权平均，再对过去 roll_days 个交易日做滚动平滑，
    作为原始因子值。根据以上计算过程，判断因子方向为正向。
    因子逻辑：股票在价格相对高位停留的时间越长，表明买方压力越大，未来收益越高；
    通过度量价格在相对高位停留的时间长短，捕捉买方压力较大的股票。

    Parameters
    ----------
    bars: pd.DataFrame
        分钟频 bars
        ['datetime', 'symbol', 'open', 'high', 'low', 'close']
    f_name: str
        因子名称，默认为 'Average_relative_price_position'
    cal_range: int
        时间窗口天数，默认为 1。研报原文分别基于1日、5日、20日三个时间窗口
        滚动计算指标，本实现采用过去 cal_range 日滚动加权平均替代。
    roll_days: int
        滚动平滑天数，默认为 20。研报原文做20个交易日的平滑得到最终因子，
        本实现采用过去 roll_days 日滚动均值替代。

    Returns
    -------
    factors: pd.DataFrame
        日频 factors
        ['datetime', 'symbol', 'factor_name', 'factor_value']
    """
    pass

bars_min = bars_1m_df.copy()
f_name = 'PV_corr'
roll_days = 20

# 分离时间轴 分钟与天
bars_min = bars_min.rename(columns = {'datetime': 'datetime_min'})
bars_min = bars_min.sort_values(['symbol', 'datetime_min'], ascending = True)
bars_min['datetime'] = bars_min['datetime_min'].dt.normalize()

# 将各个用于相关性计算的项聚合到日度表格上
bars_min['close_mul_volume'] = bars_min['close'] * bars_min['volume']
bars_min['close_pwr2'] = bars_min['close'] ** 2
bars_min['volume_pwr2'] = bars_min['volume'] ** 2

bars_daily = (
    bars_min
    .groupby(by = ['symbol', 'datetime'])
    .agg(
        N_of_Klines = ('datetime_min', 'count'),
        Sum_close_mul_volume = ('close_mul_volume', 'sum'),
        Sum_close_pwr2 = ('close_pwr2', 'sum'),
        Sum_volume_pwr2 = ('volume_pwr2', 'sum'),
        Sum_close = ('close', 'sum'),
        Sum_volume = ('volume', 'sum'),
    )
)

In [25]:
# 将各个用于相关性计算的项，滚动窗口聚合到日度上
roll_process = (
    bars_daily
    .groupby(level = 'symbol')
    .rolling(window = roll_days)
    .sum()
).reset_index(level = 0, drop = True)

# 使用变形后的相关性公式计算窗口内的 PV_corr
Numerator = (
    roll_process['N_of_Klines'] * roll_process['Sum_close_mul_volume'] -
    roll_process['Sum_close'] * roll_process['Sum_volume']
)
Denominator = np.sqrt(
    (roll_process['N_of_Klines'] * roll_process['Sum_close_pwr2'] - roll_process['Sum_close'] ** 2) *
    (roll_process['N_of_Klines'] * roll_process['Sum_volume_pwr2'] - roll_process['Sum_volume'] ** 2)
)

bars = (Numerator / Denominator).rename('PV_corr').reset_index()
bars = pd.merge(left = bars, right = market_cap.reset_index(), on = ['symbol', 'datetime'], how = 'outer')

In [30]:
import gc; gc.collect()

30

In [36]:
def _neutralize(group: pd.DataFrame) -> pd.Series:
    """
    group: 某一天（截面）的所有票
    返回: PV_corr 对 market_cap_log 回归的残差
    """

    # 1. 筛选有效行
    mask = ~ group[['PV_corr', 'market_cap_log']].isna().any(axis = 1)
    n = int(mask.sum())

    if n < 3: return pd.Series(np.nan, index = group.index, name ='PV_corr_neutral')

    x = group.loc[mask, 'market_cap_log'].values
    y = group.loc[mask, 'PV_corr'].values

    # 2. OLS 解析解
    X = np.vstack([
        np.ones(n),   # 截距项载荷
        x,            # 市值变量载荷
    ]).T
    alpha, beta, *_ = np.linalg.lstsq(X, y, rcond = None)

    # 3. 残差
    resid = y - (alpha + beta * x)

    result = pd.Series(np.nan, index = group.index, name = 'PV_corr_neutral')
    result.loc[group.loc[mask].index] = resid
    return result

# 执行（groupby apply 会自动按天切分）
bars['PV_corr_neutral'] = (
    bars.groupby('datetime')
        .apply(_neutralize)
        .reset_index(level = 0, drop = True)
)

In [37]:
bars

,symbol,datetime,PV_corr,market_cap_log,PV_corr_neutral
0,000006.XSHE,2025-11-03,NaN,NaN,NaN
1,000006.XSHE,2025-11-04,NaN,NaN,NaN
2,000006.XSHE,2025-11-05,NaN,NaN,NaN
3,000006.XSHE,2025-11-06,NaN,NaN,NaN
4,000006.XSHE,2025-11-07,NaN,NaN,NaN
...,...,...,...,...,...
243641,688776.XSHG,2026-04-24,NaN,0.0,NaN
243642,688779.XSHG,2026-04-24,NaN,0.0,NaN
243643,688789.XSHG,2026-04-24,NaN,0.0,NaN
243644,688798.XSHG,2026-04-24,NaN,0.0,NaN
